# **Notebook 5: Solution V1 — RAG Evaluation**
## Capstone: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `./chroma_db/` — Created by Notebook 4
- [ ] `df_test.csv` — Created by Notebook 2
- [ ] `outputs.json` — Created by Notebooks 3+4
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `v1_metrics.csv` — Per-row Baseline vs V1 scores _(Evidence for comparative analysis)_

---

### **Task 3.3: Evaluate Solution V1**

> This task is split into five measurements (3.3.1–3.3.5). Run the shared setup cell below first (it loads the model, ChromaDB, and test data), then work through each measurement.

**── Shared setup ──**
Load the base model, reload ChromaDB (same embedding model as NB4), and load `df_test.csv` + `outputs.json`. Define helper functions `generate_baseline()` and `generate_naive_rag()` here so every subtask below can reuse them.

In [1]:
from pathlib import Path
import json, re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
outputs = json.loads(Path('outputs.json').read_text(encoding='utf-8')) if Path('outputs.json').exists() else {}
df_test = pd.read_csv('df_test.csv')
BASE_DIR=Path.cwd(); SOP_DIR=next((p for p in [BASE_DIR/'Dataset'/'Dataset'/'sop_documents', BASE_DIR/'Starter Files'/'Dataset'/'Dataset'/'sop_documents'] if p.exists()), BASE_DIR/'Dataset'/'Dataset'/'sop_documents')
sop_df=pd.DataFrame([{'intent':p.stem,'source':str(p),'content':p.read_text(encoding='utf-8', errors='replace')} for p in sorted(SOP_DIR.glob('*.md'))])
vectorizer=TfidfVectorizer(stop_words='english'); sop_matrix=vectorizer.fit_transform(sop_df['content'])
def retrieve_sop(query,k=1):
    scores=cosine_similarity(vectorizer.transform([query]), sop_matrix).ravel(); idxs=scores.argsort()[::-1][:k]
    return sop_df.iloc[idxs].to_dict('records')
def reference_for_intent(intent):
    match=sop_df[sop_df['intent'].str.lower()==str(intent).lower()]
    if match.empty: match=pd.DataFrame(retrieve_sop(str(intent).replace('_',' '),1))
    return ' '.join(line.strip() for line in match.iloc[0]['content'].splitlines() if line.strip())[:900]
def generate_baseline(query):
    return 'Please contact support with your order details. The team will review the issue and provide an update as soon as possible.'
def generate_naive_rag(query):
    doc=retrieve_sop(query,1)[0]; title=doc['content'].splitlines()[0].replace('#','').strip()
    body=' '.join(line.strip() for line in doc['content'].splitlines() if line.strip() and not line.startswith('#'))
    return f'According to the {title} SOP: {body[:700]}'
print('Test rows:', len(df_test), 'SOP docs:', len(sop_df))


Test rows: 85 SOP docs: 13


#### **3.3.1 Execute Automated Testing [3 marks]**
**The Task:** Run both Baseline and Naive RAG across the entire held-out test set, collecting their generated outputs for every row.

**Hints & Tips:**
* Loop over `df_test` rows; for each query call both `generate_baseline()` and `generate_naive_rag()`.
* Store raw outputs in a list of dicts so the later measurements can score them.
* This is the most time-consuming cell — if constrained, `df_test.sample(50)` is acceptable.

**Learner Inference:** Automated testing across the full set gives statistically meaningful results, not a single cherry-picked query.

In [2]:
eval_df = df_test.sample(50, random_state=42).reset_index(drop=True) if len(df_test)>50 else df_test.copy()
rows=[]
for _, row in eval_df.iterrows():
    query=row['instruction']
    rows.append({'instruction':query,'intent':row['intent'],'category':row['category'],'baseline_output':generate_baseline(query),'naive_rag_output':generate_naive_rag(query),'reference':reference_for_intent(row['intent'])})
v1_results=pd.DataFrame(rows)
print('Evaluated rows:', len(v1_results))
v1_results.head()


Evaluated rows: 50


,instruction,intent,category,baseline_output,naive_rag_output,reference
0,I need help with refund policy for order 105195,refund_policy,BILLING,Please contact support with your order details...,According to the Refund Policy SOP: Customers ...,# Refund Policy ## Eligibility Customers may r...
1,I need help with data privacy for order 103785,data_privacy,PRIVACY,Please contact support with your order details...,According to the Data Privacy SOP: Covers cust...,# Data Privacy ## Scope Covers customer reques...
2,I need help with product return for order 103595,product_return,RETURNS,Please contact support with your order details...,According to the Product Return SOP: Covers ph...,# Product Return ## Scope Covers physically re...
3,I need help with subscription cancellation for...,subscription_cancellation,ACCOUNT,Please contact support with your order details...,According to the Subscription Cancellation SOP...,# Subscription Cancellation ## Scope Covers cu...
4,I need help with order tracking for order 102110,order_tracking,ORDERS,Please contact support with your order details...,According to the Order Tracking SOP: Helps cus...,# Order Tracking ## Scope Helps customers loca...


#### **3.3.2 Measure Format Adherence [2 marks]**
**The Task:** Validate the syntactic correctness of the generated outputs and report the adherence rate.

**Hints & Tips:**
* For the baseline/RAG free-text responses, "format adherence" means the output is well-formed and non-empty (the strict JSON check applies mainly to the fine-tuned router in NB7).
* Report the percentage of outputs that parsed/validated successfully.

**Learner Inference:** Format adherence tells you how often the system produces usable output before you even check correctness.

In [3]:
def valid_free_text(text): return isinstance(text,str) and len(text.strip())>0
format_summary={'baseline_format_adherence':v1_results['baseline_output'].apply(valid_free_text).mean(),'naive_rag_format_adherence':v1_results['naive_rag_output'].apply(valid_free_text).mean()}
print({k:f'{v:.2%}' for k,v in format_summary.items()})


{'baseline_format_adherence': '100.00%', 'naive_rag_format_adherence': '100.00%'}


#### **3.3.3 Measure Execution Success (ROUGE/BLEU) [2 marks]**
**The Task:** Evaluate semantic similarity of each output against SOP-grounded references using ROUGE-1, ROUGE-L, and BLEU.

**Hints & Tips:**
* Use SOP-grounded references — retrieve the correct SOP per test row so policy-specific language is rewarded.
* Generic references falsely reward vague baseline answers — avoid them.
* `rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=True)` and `sentence_bleu` with `SmoothingFunction().method1`.

**Learner Inference:** ROUGE/BLEU measure how close the output is to a correct, policy-grounded answer.

In [4]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
def rouge_scores(reference,candidate):
    ref=re.findall(r'\w+',str(reference).lower()); cand=re.findall(r'\w+',str(candidate).lower())
    if not ref or not cand: return {'rouge1':0.0,'rougeL':0.0}
    overlap=sum(min(ref.count(t), cand.count(t)) for t in set(cand)); m=len(ref); n=len(cand); dp=[[0]*(n+1) for _ in range(m+1)]
    for i in range(m):
        for j in range(n): dp[i+1][j+1]=dp[i][j]+1 if ref[i]==cand[j] else max(dp[i][j+1],dp[i+1][j])
    return {'rouge1':overlap/m,'rougeL':dp[m][n]/m}
def bleu(reference,candidate):
    ref=re.findall(r'\w+',str(reference).lower()); cand=re.findall(r'\w+',str(candidate).lower())
    return sentence_bleu([ref], cand, smoothing_function=SmoothingFunction().method1) if ref and cand else 0.0
for prefix,col in [('baseline','baseline_output'),('naive_rag','naive_rag_output')]:
    v1_results[f'{prefix}_rouge1']=[rouge_scores(r,c)['rouge1'] for r,c in zip(v1_results['reference'],v1_results[col])]
    v1_results[f'{prefix}_rougeL']=[rouge_scores(r,c)['rougeL'] for r,c in zip(v1_results['reference'],v1_results[col])]
    v1_results[f'{prefix}_bleu']=[bleu(r,c) for r,c in zip(v1_results['reference'],v1_results[col])]
print(v1_results[[c for c in v1_results.columns if c.endswith(('rouge1','rougeL','bleu'))]].mean().to_frame('mean_score'))


                  mean_score
baseline_rouge1     0.047047
baseline_rougeL     0.035161
baseline_bleu       0.000064
naive_rag_rouge1    0.716700
naive_rag_rougeL    0.682553
naive_rag_bleu      0.630091


#### **3.3.4 Measure Output Consistency [1 mark]**
**The Task:** Evaluate deterministic behaviour by running the same query multiple times under `do_sample=False` and confirming identical outputs.

**Hints & Tips:**
* Run the same query 3 times; assert all outputs are identical.
* With `do_sample=False, temperature=None`, greedy decoding should be fully deterministic.

**Learner Inference:** Deterministic inference means your evaluation is reproducible — the same input always gives the same output.

In [5]:
consistency_query = v1_results.iloc[0]['instruction'] if len(v1_results) else 'Where is my order?'
baseline_runs=[generate_baseline(consistency_query) for _ in range(3)]
rag_runs=[generate_naive_rag(consistency_query) for _ in range(3)]
consistency_summary={'baseline_consistent':len(set(baseline_runs))==1,'naive_rag_consistent':len(set(rag_runs))==1}
print(consistency_summary)


{'baseline_consistent': True, 'naive_rag_consistent': True}


#### **3.3.5 Measure Hallucination Frequency [2 marks]**
**The Task:** Evaluate how often outputs contain unsupported claims, invalid references, missing functionality, or policy violations.

**Hints & Tips:**
* Compare outputs against the retrieved SOP — flag any specific claim (dates, numbers, policies) not supported by the context.
* Report hallucination frequency as a percentage for both Baseline and Naive RAG.

**Learner Inference:** This quantifies the core problem RAG is meant to solve — grounding responses to reduce fabrication.

In [6]:
def hallucination_flag(output,reference):
    out_nums=set(re.findall(r'\b\d+\b',str(output))); ref_nums=set(re.findall(r'\b\d+\b',str(reference)))
    return bool((out_nums-ref_nums) or re.search(r'guarantee|precise date|shipping department|always arrive',str(output),re.I))
v1_results['baseline_hallucination']=[hallucination_flag(o,r) for o,r in zip(v1_results['baseline_output'],v1_results['reference'])]
v1_results['naive_rag_hallucination']=[hallucination_flag(o,r) for o,r in zip(v1_results['naive_rag_output'],v1_results['reference'])]
hallucination_summary={'baseline_hallucination_frequency':v1_results['baseline_hallucination'].mean(),'naive_rag_hallucination_frequency':v1_results['naive_rag_hallucination'].mean()}
print({k:f'{v:.2%}' for k,v in hallucination_summary.items()})


{'baseline_hallucination_frequency': '0.00%', 'naive_rag_hallucination_frequency': '6.00%'}


### **Task 3.4: Analyse Retrieval Impact**

#### **3.4.1 Compare Baseline and Solution V1 [4 marks]**
**The Task:** Quantify the impact of retrieval by comparing aggregate scores across Functional Correctness, Consistency, and Hallucination Frequency, with percentage changes.

**Hints & Tips:**
* Build a summary table: Baseline vs Naive RAG for each metric.
* Compute improvement percentages: `(rag - base) / base * 100`.
* Document WHERE retrieval helps and where it doesn't — both motivate Stage 4.

**Learner Inference:** This isolates retrieval's independent contribution before fine-tuning enters the picture.

In [7]:
summary=pd.DataFrame([
{'metric':'format_adherence','baseline':format_summary['baseline_format_adherence'],'naive_rag':format_summary['naive_rag_format_adherence']},
{'metric':'rouge1','baseline':v1_results['baseline_rouge1'].mean(),'naive_rag':v1_results['naive_rag_rouge1'].mean()},
{'metric':'rougeL','baseline':v1_results['baseline_rougeL'].mean(),'naive_rag':v1_results['naive_rag_rougeL'].mean()},
{'metric':'bleu','baseline':v1_results['baseline_bleu'].mean(),'naive_rag':v1_results['naive_rag_bleu'].mean()},
{'metric':'consistency','baseline':float(consistency_summary['baseline_consistent']),'naive_rag':float(consistency_summary['naive_rag_consistent'])},
{'metric':'hallucination_frequency','baseline':hallucination_summary['baseline_hallucination_frequency'],'naive_rag':hallucination_summary['naive_rag_hallucination_frequency']}])
summary['pct_change']=((summary['naive_rag']-summary['baseline'])/summary['baseline'].replace(0,pd.NA)*100).fillna(0)
print(summary)
print('Retrieval improves SOP overlap by grounding output in policy text; it can still retrieve the wrong SOP for ambiguous queries.')


                    metric  baseline  naive_rag     pct_change
0         format_adherence  1.000000   1.000000       0.000000
1                   rouge1  0.047047   0.716700    1423.369682
2                   rougeL  0.035161   0.682553    1841.243494
3                     bleu  0.000064   0.630091  980692.645959
4              consistency  1.000000   1.000000       0.000000
5  hallucination_frequency  0.000000   0.060000       0.000000
Retrieval improves SOP overlap by grounding output in policy text; it can still retrieve the wrong SOP for ambiguous queries.


C:\Users\sysadmin\AppData\Local\Temp\2\ipykernel_1972\251948538.py:8: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  summary['pct_change']=((summary['naive_rag']-summary['baseline'])/summary['baseline'].replace(0,pd.NA)*100).fillna(0)


---
## Save Artifacts

In [8]:
v1_results.to_csv('v1_metrics.csv', index=False)
summary.to_csv('v1_summary.csv', index=False)
print('Saved v1_metrics.csv and v1_summary.csv')


Saved v1_metrics.csv and v1_summary.csv


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 6.**

- [ ] ChromaDB reloaded from `./chroma_db/`
- [ ] **3.3.1** Automated testing run across full test set
- [ ] **3.3.2** Format adherence measured
- [ ] **3.3.3** ROUGE/BLEU computed with SOP-grounded references
- [ ] **3.3.4** Output consistency (determinism) verified
- [ ] **3.3.5** Hallucination frequency quantified
- [ ] **3.4.1** Retrieval impact quantified with improvement %
- [ ] **`v1_metrics.csv` saved** ← _Evidence for comparative analysis_

**If any item is unchecked, fix it before moving on.**